In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../trans_new.csv')
if not df.empty:
    print(f'Loaded {len(df)} rows from csv dataset')

Loaded 538643 rows from csv dataset


In [3]:
df.dtypes

WEEK_END_DATE                  str
STORE_ID                     int64
UPC                          int64
UNITS                        int64
VISITS                       int64
HHS                          int64
SPEND                      float64
PRICE                      float64
BASE_PRICE                 float64
FEATURE                      int64
DISPLAY                      int64
TPR_ONLY                     int64
DESCRIPTION                    str
MANUFACTURER                   str
CATEGORY                       str
SUB_CATEGORY                   str
PRODUCT_SIZE                   str
STORE_NAME                     str
ADDRESS_CITY_NAME              str
ADDRESS_STATE_PROV_CODE        str
MSA_CODE                     int64
SEG_VALUE_NAME                 str
PARKING_SPACE_QTY          float64
SALES_AREA_SIZE_NUM          int64
AVG_WEEKLY_BASKETS         float64
dtype: object

In [4]:
print(f'Available columns are {df.columns}')
print(f'{df["SEG_VALUE_NAME"].value_counts(normalize=True)}')

Available columns are Index(['WEEK_END_DATE', 'STORE_ID', 'UPC', 'UNITS', 'VISITS', 'HHS', 'SPEND',
       'PRICE', 'BASE_PRICE', 'FEATURE', 'DISPLAY', 'TPR_ONLY', 'DESCRIPTION',
       'MANUFACTURER', 'CATEGORY', 'SUB_CATEGORY', 'PRODUCT_SIZE',
       'STORE_NAME', 'ADDRESS_CITY_NAME', 'ADDRESS_STATE_PROV_CODE',
       'MSA_CODE', 'SEG_VALUE_NAME', 'PARKING_SPACE_QTY',
       'SALES_AREA_SIZE_NUM', 'AVG_WEEKLY_BASKETS'],
      dtype='str')
SEG_VALUE_NAME
MAINSTREAM    0.560310
UPSCALE       0.226178
VALUE         0.213512
Name: proportion, dtype: float64


In [5]:
# Clean 'OZ' / 'FL OZ' case-insensitively and safely convert to float
df["PRODUCT_SIZE"] = pd.to_numeric(
    df["PRODUCT_SIZE"]
    .str.replace(r"(?i)\s*(fl\s*)?oz", "", regex=True)
    .str.strip(),
    errors="coerce"
)


In [6]:
# 2. Compute Overall Volume for the row
df['OVERALL_VOLUME'] = df['UNITS'] * df['PRODUCT_SIZE']

# 3. Compute Price Per Volume (PPV)
df['PPV'] = df['SPEND'] / df['OVERALL_VOLUME']

In [14]:
sub_category_summary = (
    df[df["BASE_PRICE"] > 0]
    .groupby("SUB_CATEGORY")["BASE_PRICE"]
    .mean()
    .reset_index(name="avg_base_price")
)
sub_category_summary

,SUB_CATEGORY,avg_base_price
0,ADULT CEREAL,2.738438
1,ALL FAMILY CEREAL,2.988374
2,KIDS CEREAL,2.854965
3,MOUTHWASH/RINSES AND SPRAYS,3.443935
4,PIZZA/PREMIUM,5.684227
5,PRETZELS,2.284943


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

df=df.dropna()

# 1. Base Price per Unit volume
# (Assumes PRODUCT_SIZE is cleaned into a numeric volume/weight column)
df['UNIT_BASE_PRICE'] = df['BASE_PRICE'] / df['PRODUCT_SIZE']

# 2. Aggregate to UPC level within a Sub-Category
upc_summary = df.groupby(['SUB_CATEGORY', 'UPC']).agg(
    median_unit_price=('UNIT_BASE_PRICE', 'median'),
    total_units=('UNITS', 'sum'),
    PPV=('PPV','mean')
).reset_index()

# 3. Algorithm: Run Tiering per Sub-Category
def apply_kmeans_tiers(group, n_tiers=3):
    if len(group) < n_tiers:
        group['PRICE_TIER'] = 'Standard'
        return group
    
    # Fit K-Means on 1D Price per Volume Data
    X = group[['PPV']].values
    kmeans = KMeans(n_clusters=n_tiers, random_state=42, n_init=10).fit(X)
    
    # Order clusters by their center prices to map to text tiers
    centers = kmeans.cluster_centers_.flatten()
    sorted_tier_mapping = {old_label: new_label for new_label, old_label in enumerate(np.argsort(centers))}
    
    raw_labels = kmeans.labels_
    ordered_labels = [sorted_tier_mapping[label] for label in raw_labels]
    
    # Map numbers to descriptive names
    tier_names = {0: 'Value', 1: 'Standard', 2: 'Premium', 3: 'Super Premium'}
    group['PRICE_TIER'] = [tier_names.get(i, 'Premium') for i in ordered_labels]
    
    return group

# Apply the algorithm across all categories
tiered_portfolio = upc_summary.groupby('SUB_CATEGORY', group_keys=False).apply(apply_kmeans_tiers)


In [8]:
tiered_portfolio["PRICE_TIER"].value_counts(normalize="True")

PRICE_TIER
Standard    0.44186
Value       0.27907
Premium     0.27907
Name: proportion, dtype: float64

In [9]:
# Function to assign price tiers based on PPV within each sub-category
def assign_price_tiers(group):
    # Calculate 33rd and 66th percentiles within sub-category
    q33 = group['PPV'].quantile(0.33)
    q66 = group['PPV'].quantile(0.66)
    
    conditions = [
        group['PPV'] <= q33,
        (group['PPV'] > q33) & (group['PPV'] <= q66),
        group['PPV'] > q66
    ]
    labels = ['Value', 'Mainstream', 'Premium']
    
    group['PRICE_TIER'] = np.select(conditions, labels, default='Mainstream')
    return group

# Apply tiering by sub-category
upc_tiered_df = upc_summary.groupby('SUB_CATEGORY', group_keys=False).apply(assign_price_tiers)
upc_tiered_df["PRICE_TIER"].value_counts(normalize="True")

PRICE_TIER
Value         0.348837
Premium       0.348837
Mainstream    0.302326
Name: proportion, dtype: float64

In [18]:
import numpy as np
import pandas as pd
import pulp


def derive_tactic_parameters_from_df(df, target_upc):
    """Extracts baseline volume, regular price, and empirical lift factors for

    a given UPC directly from your dataset.
    """
    # Filter dataset for target UPC
    upc_df = df[df["UPC"] == target_upc].copy()

    # 1. Clean & derive promo tactic column
    conditions = [
        (upc_df["FEATURE"] == 1) & (upc_df["DISPLAY"] == 1),
        (upc_df["FEATURE"] == 1) & (upc_df["DISPLAY"] == 0),
        (upc_df["DISPLAY"] == 1) & (upc_df["FEATURE"] == 0),
        (upc_df["TPR_ONLY"] == 1)
        & (upc_df["FEATURE"] == 0)
        & (upc_df["DISPLAY"] == 0),
    ]
    tactics = ["FEAT_AND_DISP", "FEATURE_ONLY", "DISPLAY_ONLY", "TPR_ONLY"]
    upc_df["PROMO_TACTIC"] = np.select(conditions, tactics, default="NO_PROMO")

    # 2. Extract Base Parameters
    base_price = upc_df["BASE_PRICE"].median()
    no_promo_df = upc_df[upc_df["PROMO_TACTIC"] == "NO_PROMO"]
    base_units = (
        no_promo_df["UNITS"].mean()
        if len(no_promo_df) > 0
        else upc_df["UNITS"].mean()
    )

    # 3. Derive Lift Multiplier and Average Discount Depth by Tactic
    tactic_specs = {}

    # Define standard fixed setup/slotting costs per tactic (business assumptions)
    fixed_costs = {
        "NO_PROMO": 0,
        "TPR_ONLY": 200,  # Shelf tag / POS setup cost
        "FEATURE_ONLY": 1200,  # Circular / digital ad fee
        "DISPLAY_ONLY": 1500,  # Endcap display fee
        "FEAT_AND_DISP": 2500,  # Combined co-op fee
    }

    all_tactics = [
        "NO_PROMO",
        "TPR_ONLY",
        "FEATURE_ONLY",
        "DISPLAY_ONLY",
        "FEAT_AND_DISP",
    ]

    for t in all_tactics:
        t_data = upc_df[upc_df["PROMO_TACTIC"] == t]

        if len(t_data) > 0 and t != "NO_PROMO":
            lift = t_data["UNITS"].mean() / max(1, base_units)
            avg_realized_price = t_data["PRICE"].mean()
            discount = max(0.0, (base_price - avg_realized_price) / base_price)
        elif t == "NO_PROMO":
            lift = 1.0
            discount = 0.0
        else:
            # Fallback estimates if a tactic wasn't historically executed for this UPC
            fallback_lifts = {
                "TPR_ONLY": 1.3,
                "FEATURE_ONLY": 1.6,
                "DISPLAY_ONLY": 1.9,
                "FEAT_AND_DISP": 2.8,
            }
            fallback_discounts = {
                "TPR_ONLY": 0.15,
                "FEATURE_ONLY": 0.10,
                "DISPLAY_ONLY": 0.10,
                "FEAT_AND_DISP": 0.20,
            }
            lift = fallback_lifts.get(t, 1.0)
            discount = fallback_discounts.get(t, 0.0)

        tactic_specs[t] = {
            "lift": round(lift, 2),
            "discount": round(discount, 3),
            "fixed_cost": fixed_costs[t],
        }

    return base_units, base_price, tactic_specs


def optimize_tpo_calendar(
    df,
    target_upc,
    unit_cost=1.50,
    trade_budget=5000.0,
    max_promo_weeks=4,
    cooldown_weeks=2,
):
    """Builds and solves the Trade Promotion Calendar MILP model."""
    # Derive inputs dynamically from dataset
    base_units, base_price, tactic_specs = derive_tactic_parameters_from_df(
        df, target_upc
    )

    weeks = list(range(1, 13))  # 12-week planning horizon
    tactics = list(tactic_specs.keys())

    # Pre-compute economics matrices per tactic execution
    profit_matrix = {}
    spend_matrix = {}
    units_matrix = {}

    for t in tactics:
        spec = tactic_specs[t]
        units = base_units * spec["lift"]
        realized_price = base_price * (1.0 - spec["discount"])

        # Gross Profit = Realized Margin - Fixed Execution Cost
        margin_per_unit = realized_price - unit_cost
        gross_profit = (units * margin_per_unit) - spec["fixed_cost"]

        # Trade Spend = Markdown Subsidy + Fixed Execution Fee
        trade_spend = ((base_price - realized_price) * units) + spec[
            "fixed_cost"
        ]

        profit_matrix[t] = gross_profit
        spend_matrix[t] = trade_spend
        units_matrix[t] = units

    # Initialize PuLP Model
    model = pulp.LpProblem("TPO_Calendar_Optimization", pulp.LpMaximize)

    # Decision Variable: X[w, t] = 1 if tactic t is scheduled in week w
    x = pulp.LpVariable.dicts("Tactic", (weeks, tactics), cat=pulp.LpBinary)

    # Objective: Maximize 12-Week Gross Margin
    model += pulp.lpSum(
        [x[w][t] * profit_matrix[t] for w in weeks for t in tactics]
    )

    # Constraint 1: Exactly 1 tactic per week
    for w in weeks:
        model += pulp.lpSum([x[w][t] for t in tactics]) == 1

    # Constraint 2: Total Trade Budget Limit
    model += (
        pulp.lpSum([x[w][t] * spend_matrix[t] for w in weeks for t in tactics])
        <= trade_budget
    )

    # Constraint 3: Max Promotional Frequency Cap
    model += (
        pulp.lpSum(
            [x[w][t] for w in weeks for t in tactics if t != "NO_PROMO"]
        )
        <= max_promo_weeks
    )

    # Constraint 4: Cooldown period after promotional weeks
    for w in weeks:
        for offset in range(1, cooldown_weeks + 1):
            if w + offset <= max(weeks):
                promo_w = pulp.lpSum(
                    [x[w][t] for t in tactics if t != "NO_PROMO"]
                )
                promo_future = pulp.lpSum(
                    [x[w + offset][t] for t in tactics if t != "NO_PROMO"]
                )
                model += promo_w + promo_future <= 1

    # Solve model
    solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    # Extract Results
    output_schedule = []
    for w in weeks:
        for t in tactics:
            if x[w][t].varValue == 1:
                output_schedule.append({
                    "Week": w,
                    "Tactic": t,
                    "Units": round(units_matrix[t]),
                    "Trade_Spend": round(spend_matrix[t], 2),
                    "Gross_Profit": round(profit_matrix[t], 2),
                })

    return pd.DataFrame(output_schedule), pulp.value(model.objective)


# ==========================================
# EXAMPLE EXECUTION WITH YOUR SCHEMA
# ==========================================
# (Assuming 'df' is loaded with your actual dataset)
schedule_df, total_profit = optimize_tpo_calendar(df, target_upc=1111085319, unit_cost=1.85, trade_budget=6000)
print(schedule_df)

    Week    Tactic  Units  Trade_Spend  Gross_Profit
0      1  NO_PROMO     18          0.0         -0.18
1      2  NO_PROMO     18          0.0         -0.18
2      3  NO_PROMO     18          0.0         -0.18
3      4  NO_PROMO     18          0.0         -0.18
4      5  NO_PROMO     18          0.0         -0.18
5      6  NO_PROMO     18          0.0         -0.18
6      7  NO_PROMO     18          0.0         -0.18
7      8  NO_PROMO     18          0.0         -0.18
8      9  NO_PROMO     18          0.0         -0.18
9     10  NO_PROMO     18          0.0         -0.18
10    11  NO_PROMO     18          0.0         -0.18
11    12  NO_PROMO     18          0.0         -0.18
